# Pneumonia Chest X-ray Classification

End-to-end transfer-learning pipeline based on the project brief: classify chest X-rays as **Pneumonia** or **Normal**. The notebook supports directory datasets with `train/`, `val/`/`validation/`, and `test/` folders, or creates a stratified split when only class folders are available.

> Research/education use only. This model is not a medical diagnostic device. Evaluate on an independent, patient-level dataset before any clinical use.

## 1. Configuration

Expected layouts:
- `DATA_DIR/train/NORMAL`, `DATA_DIR/train/PNEUMONIA`
- optional `DATA_DIR/val/...` and `DATA_DIR/test/...`

If the downloaded archive has a different layout, set `DATA_DIR` to the directory containing the class folders. You can also set `ZIP_PATH` to the archive and run the download cell.

In [ ]:
# Install once if needed
# %pip install -q tensorflow scikit-learn seaborn pandas matplotlib pillow

from pathlib import Path
import os, random, zipfile, shutil, json, numpy as np, pandas as pd
SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

DATA_DIR = Path("C:\\Users\\govin\\OneDrive\\Desktop\\1012\\Chest-Xray-2")   # change this
ZIP_PATH = None                       # e.g. "data/Chest-Xray-2.zip"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 8
EPOCHS_FINE = 15
AUTOTUNE = __import__("tensorflow").data.AUTOTUNE

if ZIP_PATH and Path(ZIP_PATH).exists() and not DATA_DIR.exists():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DATA_DIR.parent)
print("DATA_DIR:", DATA_DIR.resolve())

ModuleNotFoundError: No module named 'tensorflow'

## 2. Imports and dataset discovery

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt, seaborn as sns

print(tf.__version__)

def find_class_dirs(root):
    dirs = [p for p in root.rglob("*") if p.is_dir() and p.name.upper() in {"NORMAL", "PNEUMONIA"}]
    return dirs

def image_paths(root):
    exts={".jpg",".jpeg",".png",".bmp",".webp"}
    rows=[]
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            label = 1 if "PNEUMONIA" in {x.upper() for x in p.parts} else 0
            rows.append((str(p), label))
    return pd.DataFrame(rows, columns=["path","label"]).drop_duplicates()

all_df = image_paths(DATA_DIR)
if all_df.empty:
    raise FileNotFoundError("No images found. Set DATA_DIR to the extracted dataset directory.")
print(all_df.label.value_counts().rename(index={0:"NORMAL",1:"PNEUMONIA"}))

## 3. Patient-safe-ish stratified split

The split is stratified by class. If patient identifiers are available in filenames, replace this with a patient-level split to prevent leakage. Existing `test` folders are kept separate when detected.

In [ ]:
def subset_df(root, names):
    mask = all_df.path.map(lambda x: any(n.lower() in {p.lower() for p in Path(x).parts} for n in names))
    return all_df[mask].copy()

test_df = subset_df(DATA_DIR, ["test", "testing"])
train_pool = all_df.drop(index=test_df.index) if len(test_df) else all_df
if len(test_df)==0:
    train_pool, test_df = train_test_split(train_pool, test_size=0.15, stratify=train_pool.label, random_state=SEED)
val_df = subset_df(Path(str(DATA_DIR)), ["val", "validation"])
if len(val_df):
    train_df = train_pool.drop(index=val_df.index, errors="ignore")
else:
    train_df, val_df = train_test_split(train_pool, test_size=0.15, stratify=train_pool.label, random_state=SEED)
for name,df in [("train",train_df),("val",val_df),("test",test_df)]:
    print(name, len(df), df.label.value_counts().to_dict())

## 4. Input pipelines and augmentation

In [ ]:
def make_ds(df, training=False):
    paths=df.path.values; labels=df.label.values.astype("float32")
    ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    if training: ds=ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    def load(path,label):
        x=tf.io.read_file(path); x=tf.image.decode_image(x, channels=3, expand_animations=False)
        x=tf.image.resize(x, IMG_SIZE); x=tf.cast(x,tf.float32)
        return x,label
    return ds.map(load,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df,True), make_ds(val_df), make_ds(test_df)

augment = keras.Sequential([
    layers.RandomRotation(0.03), layers.RandomZoom(0.08),
    layers.RandomTranslation(0.03,0.03), layers.RandomContrast(0.10)
], name="augmentation")


## 5. Transfer-learning model

EfficientNetB0 provides a strong baseline for limited medical-image datasets. ImageNet weights are used only as initialization; the final classifier is trained for this binary task.

In [ ]:
base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE,3))
base.trainable=False
inputs=keras.Input(shape=(*IMG_SIZE,3))
x=augment(inputs)
x=keras.applications.efficientnet.preprocess_input(x)
x=base(x, training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dropout(0.35)(x)
outputs=layers.Dense(1, activation="sigmoid")(x)
model=keras.Model(inputs,outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc"), keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])
model.summary()

## 6. Class weighting and training

In [ ]:
counts=train_df.label.value_counts().to_dict(); n=len(train_df)
class_weight={c:n/(2*counts.get(c,1)) for c in [0,1]}
callbacks=[keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True), keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=.3, patience=2), keras.callbacks.ModelCheckpoint("best_pneumonia.keras", monitor="val_auc", mode="max", save_best_only=True)]
hist1=model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=callbacks)

# Fine-tune the last part of the backbone
base.trainable=True
for layer in base.layers[:-30]: layer.trainable=False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc"), keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])
hist2=model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, class_weight=class_weight, callbacks=callbacks)
model=keras.models.load_model("best_pneumonia.keras")

## 7. Evaluation

Accuracy alone can be misleading when class frequencies differ. Report sensitivity/recall, specificity, precision, F1, ROC-AUC, and the confusion matrix.

In [ ]:
y_true=test_df.label.to_numpy()
y_prob=model.predict(test_ds).ravel()
y_pred=(y_prob>=0.5).astype(int)
print(classification_report(y_true,y_pred,target_names=["NORMAL","PNEUMONIA"],digits=4))
print("ROC-AUC:", round(roc_auc_score(y_true,y_prob),4))
cm=confusion_matrix(y_true,y_pred)
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["NORMAL","PNEUMONIA"],yticklabels=["NORMAL","PNEUMONIA"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

fpr,tpr,_=roc_curve(y_true,y_prob)
plt.plot(fpr,tpr,label=f"AUC={roc_auc_score(y_true,y_prob):.3f}"); plt.plot([0,1],[0,1],"--"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.legend(); plt.show()

## 8. Threshold selection

For screening, missing pneumonia may be more costly than a false alarm. Select a threshold on validation data according to the intended use; do not tune it on the test set.

In [ ]:
val_true=val_df.label.to_numpy(); val_prob=model.predict(val_ds).ravel()
rows=[]
for t in np.arange(.10,.91,.05):
    p=(val_prob>=t).astype(int); tn,fp,fn,tp=confusion_matrix(val_true,p,labels=[0,1]).ravel()
    rows.append([t,tp/(tp+fn+1e-9),tn/(tn+fp+1e-9),tp/(tp+fp+1e-9)])
thresholds=pd.DataFrame(rows,columns=["threshold","sensitivity","specificity","precision"])
thresholds.sort_values("sensitivity",ascending=False).head(10)

## 9. Single-image inference and export

In [ ]:
def predict_xray(path, threshold=0.5):
    img=tf.keras.utils.load_img(path,target_size=IMG_SIZE)
    arr=tf.keras.utils.img_to_array(img)[None,...]
    prob=float(model.predict(arr,verbose=0)[0,0])
    return {"path":str(path),"pneumonia_probability":prob,"prediction":"PNEUMONIA" if prob>=threshold else "NORMAL","threshold":threshold}

# Example:
# predict_xray("path/to/chest_xray.jpeg", threshold=0.5)
model.save("pneumonia_efficientnetb0.keras")
with open("class_names.json","w") as f: json.dump({"0":"NORMAL","1":"PNEUMONIA"},f)
print("Saved model and class names")

## 10. Practical quality checklist

- Confirm that no patient appears in more than one split.
- Use an external dataset from a different source for generalization testing.
- Inspect false positives and false negatives; check for view, marker, device, and demographic shortcuts.
- Calibrate probabilities and select a threshold with clinical stakeholders.
- Document dataset provenance, preprocessing, model version, and limitations.
- Do not use the output as a standalone diagnosis.